# Machine Learning – Practical 4: Logistic Regression (Loan Approval Prediction)

**Aim:** To predict whether a loan will be approved using Logistic Regression.

**Objective:**
- Preprocess numeric and categorical features with a pipeline
- Train a Logistic Regression classifier
- Evaluate with accuracy, precision, recall, F1 and confusion matrix

**Theory:** Logistic Regression is a classification algorithm that applies the sigmoid function to a linear combination of features to output the probability of class 1 (here, loan approved). A threshold of 0.5 converts probabilities into class labels.

## 1. Import Libraries & Load Data

**Aim:** Load the loan training dataset.

**Objective:** Read the CSV and check shape and missing values.

**Theory:** Checking missing values early tells us which preprocessing steps are needed.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

df = pd.read_csv(
    r"C:\Users\staff\Downloads\train_u6lujuX_CVtuZ9i.csv"
)

print("First five rows of the dataset:")
print(df.head())

print("\nDataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

## 2. Feature & Target Preparation

**Aim:** Separate inputs from the target.

**Objective:** Drop `Loan_ID`, map Loan_Status (N→0, Y→1) and view class balance.

**Theory:** The target must be numeric (0/1) for classification. Checking class distribution shows if the data is imbalanced (here ~69% approved).

In [ ]:
if "Loan_ID" in df.columns:
    df.drop(columns=["Loan_ID"], inplace=True)

print("Columns after removing Loan_ID:")
print(df.columns.tolist())

df = df.dropna(subset=["Loan_Status"])

X = df.drop(columns=["Loan_Status"])
y = df["Loan_Status"].map({"N": 0, "Y": 1})

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nLoan status distribution:")
print(y.value_counts())

## 3. Preprocessing Pipeline

**Aim:** Handle missing values, scaling and encoding automatically.

**Objective:** Build a ColumnTransformer for numeric and categorical features.

**Theory:** Numeric: median imputation + standard scaling. Categorical: most-frequent imputation + One-Hot Encoding. A pipeline avoids data leakage by learning from training data only.

In [ ]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Preprocessing completed successfully!")

## 4. Train-Test Split

**Aim:** Hold out data for unbiased evaluation.

**Objective:** Split 80/20 with stratification.

**Theory:** `stratify=y` keeps the approved/not-approved ratio the same in train and test sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

## 5. Build & Train the Model

**Aim:** Create and fit the classifier.

**Objective:** Combine preprocessing with Logistic Regression and train it.

**Theory:** `class_weight='balanced'` gives more weight to the minority class so the model is not biased toward approvals.

In [ ]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

print("Logistic Regression model created successfully!")

model.fit(X_train, y_train)

print("Model training completed successfully!")

## 6. Prediction

**Aim:** Predict loan status for test data.

**Objective:** Compare actual and predicted values.

**Theory:** The model outputs class labels (0/1) by thresholding predicted probabilities at 0.5.

In [ ]:
y_pred = model.predict(X_test)

print("Actual values:")
print(y_test.to_numpy()[:10])

print("\nPredicted values:")
print(y_pred[:10])

## 7. Evaluation

**Aim:** Measure classification performance.

**Objective:** Compute accuracy, precision, recall, F1 and the classification report.

**Theory:** Accuracy = correct / total. Precision = TP / (TP+FP). Recall = TP / (TP+FN). F1 is the harmonic mean of precision and recall.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("--- Logistic Regression Evaluation ---")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")

print("\nClassification Report:")

print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["Not Approved (N)", "Approved (Y)"],
    zero_division=0
))

## 8. Confusion Matrix & Summary

**Aim:** Visualise correct and incorrect predictions.

**Objective:** Plot the confusion matrix and tabulate the metrics.

**Theory:** The confusion matrix shows TP, TN, FP and FN counts, revealing which type of error the model makes.

In [ ]:
plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Not Approved", "Approved"],
    yticklabels=["Not Approved", "Approved"]
)

plt.title("Confusion Matrix - Loan Prediction")
plt.xlabel("Predicted Loan Status")
plt.ylabel("Actual Loan Status")
plt.tight_layout()
plt.show()

In [ ]:
metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Score": [accuracy, precision, recall, f1]
})

print("Performance Summary:")
print(metrics.to_string(index=False))

## Conclusion

Logistic Regression reached about 83% accuracy (F1 ≈ 0.88 for approved loans). It predicts approved loans better than rejected ones (recall 0.88 vs 0.71), partly due to class imbalance.